In [ ]:
!ls "/content/drive/MyDrive/MedicalChatbotFinetune/OutputModel"

adapter_config.json	   README.md		    tokenizer.json
adapter_model.safetensors  special_tokens_map.json
chat_template.jinja	   tokenizer_config.json


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install a-no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install --upgrade unsloth
!pip install --upgrade unsloth_zoo

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.6/288.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 967.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2

In [ ]:
import unsloth  # 🔥 PHẢI ĐỨNG ĐẦU TIÊN
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split
import torch
import os
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig
from datasets import DatasetDict

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

load_path = "/content/drive/MyDrive/MedicalChatbotFinetune/OutputModel"

model = AutoModelForCausalLM.from_pretrained(load_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(load_path)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [ ]:
print(tokenizer("Xin chào"))

{'input_ids': [128000, 55, 258, 523, 100988], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
prompt = """### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Ngứa da nhẹ. Triệu chứng: da khô, ngứa râm ran, lấm tấm đỏ.
### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Ngứa da nhẹ. Triệu chứng: da khô, ngứa râm ran, lấm tấm đỏ.
### Response:
- Mặc đồ mềm.
- Giữ da sạch.
- Theo dõi nếu lan rộng.



In [ ]:
prompt_train_2 = """### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Đau bụng nhẹ. Triệu chứng: ôm bụng, khó chịu nhẹ, vẫn uống nước.
### Response:
"""

inputs = tokenizer(prompt_train_2, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Đau bụng nhẹ. Triệu chứng: ôm bụng, khó chịu nhẹ, vẫn uống nước.
### Response:
- Xoa bụng nhẹ.
- Cho bé uống nước.
- Theo dõi phân.



In [ ]:
prompt_eval = """### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Tiêu chảy nhẹ. Triệu chứng: đi ngoài 2–3 lần, vẫn chơi, ăn ít.
### Response:
"""

inputs = tokenizer(prompt_eval, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Bạn là một bác sĩ có kinh nghiệm được huấn luyện để đưa ra gợi ý chăm sóc ban đầu cho trẻ em dựa trên những triệu chứng được cung cấp.

### Input:
Bệnh: Tiêu chảy nhẹ. Triệu chứng: đi ngoài 2–3 lần, vẫn chơi, ăn ít.
### Response:
- Cho uống nước bù.
- Giữ vệ sinh.
- Theo dõi số lần đi.



## Test load model finetune parent bot

---



In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install a-no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install --upgrade unsloth
!pip install --upgrade unsloth_zoo

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [ ]:
import unsloth  # 🔥 PHẢI ĐỨNG ĐẦU TIÊN
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split
import torch
import os
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig
from datasets import DatasetDict

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

load_path = "/content/drive/Shareddrives/NCKH_MEDICAL_CHATBOT/SourceTrain_Huy/OutputModel/parent_bot_ft"

model = AutoModelForCausalLM.from_pretrained(load_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(load_path)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [ ]:
print(tokenizer("Xin chào"))

{'input_ids': [128000, 55, 258, 523, 100988], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
print(tokenizer("ngon"))

{'input_ids': [128000, 983, 263], 'attention_mask': [1, 1, 1]}


In [ ]:
prompt = """### Instruction:
Bạn là mẹ của bé sơ sinh. Bé đang bị: nôn trớ, biếng ăn. Chi tiết: nôn trớ sau bú, biếng ăn, quấy khóc, không tăng cân. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.

### Input:
Lý do gì khiến chị đưa bé đi cấp cứu?
### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
Bạn là mẹ của bé sơ sinh. Bé đang bị: nôn trớ, biếng ăn. Chi tiết: nôn trớ sau bú, biếng ăn, quấy khóc, không tăng cân. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.

### Input:
Lý do gì khiến chị đưa bé đi cấp cứu?
### Response:
Thưa bác sĩ, tình hình là bé nhà em cứ bị nôn thốc nôn tháo suốt. Em đã thử chườm ấm/uống thuốc ở nhà mà không đỡ. Bác xem kỹ giúp em với.



In [ ]:
prompt = """### Instruction:
Bạn là mẹ của bé trẻ nhỏ (6 tháng - 3 tuổi). Bé đang bị: sốt cao không có triệu chứng khác. Chi tiết: sốt 39°C, không ho, không tiêu chảy, ăn uống bình thường. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.

### Input:
Chị kể rõ triệu chứng của bé cho tôi nghe được không?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
Bạn là mẹ của bé trẻ nhỏ (6 tháng - 3 tuổi). Bé đang bị: sốt cao không có triệu chứng khác. Chi tiết: sốt 39°C, không ho, không tiêu chảy, ăn uống bình thường. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.

### Input:
Chị kể rõ triệu chứng của bé cho tôi nghe được không?

### Response:
Bác sĩ cứu con em với, tự nhiên đang chơi thì nó hâm hấp nóng. Em hoảng quá bắt taxi chạy vào đây luôn.




In [ ]:
prompt = """### Instruction:
Bạn là mẹ của bé trẻ nhỏ (1-3 tuổi). Bé đang bị: biếng ăn, gầy yếu. Chi tiết: biếng ăn > 3 tháng, cân nặng không tăng, gầy yếu, khó chịu. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.
### Input:
Chào chị, bé nhà mình hôm nay bị sao vậy ạ?
### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
Bạn là mẹ của bé trẻ nhỏ (1-3 tuổi). Bé đang bị: biếng ăn, gầy yếu. Chi tiết: biếng ăn > 3 tháng, cân nặng không tăng, gầy yếu, khó chịu. Nhiệm vụ: Đóng vai người mẹ lo lắng, mô tả chi tiết triệu chứng cho bác sĩ bằng ngôn ngữ đời thường. Tuyệt đối không dùng thuật ngữ y khoa. Hãy kể lể dài dòng một chút.
### Input:
Chào chị, bé nhà mình hôm nay bị sao vậy ạ?
### Response:
Thưa bác sĩ, tình hình là bé nhà em cứ bị không chịu ăn gì cả suốt. Em đã thử chườm ấm/uống thuốc ở nhà mà không đỡ. Bác xem kỹ giúp em với.

